# 📊 HealthProcessAI - R Implementation (Google Colab)

**Healthcare Process Mining with LLM Integration - R Version**

**Developed at SMAILE (Stockholm Medical Artificial Intelligence and Learning Environments), Karolinska Institutet**

This notebook demonstrates the R implementation of HealthProcessAI, featuring:
- bupaR ecosystem for process mining
- httr2 for modern HTTP/API interaction
- R6 classes for object-oriented programming
- RMarkdown for report generation
- Multi-model LLM orchestration via OpenRouter

---

## 🔧 Step 1: Environment Setup

First, we'll configure the R environment in Colab and install required packages.

In [ ]:
# Enable R kernel in Colab
%load_ext rpy2.ipython

In [ ]:
%%R
# Install required R packages
install_packages <- function(packages) {
  new_packages <- packages[!(packages %in% installed.packages()[,"Package"])]
  if(length(new_packages) > 0) {
    install.packages(new_packages, repos = "https://cloud.r-project.org/", quiet = TRUE)
  }
}

# Core packages
core_packages <- c(
  "tidyverse",    # Data manipulation
  "lubridate",    # Date/time handling
  "glue",         # String formatting
  "jsonlite",     # JSON handling
  "R6",           # Object-oriented programming
  "httr2"         # Modern HTTP client
)

# Process mining packages
pm_packages <- c(
  "bupaR",        # Process mining core
  "edeaR",        # Exploratory data analysis
  "processmapR",  # Process map visualization
  "petrinetR",    # Petri net analysis
  "eventdataR"    # Event data repository
)

# Analytics packages
analytics_packages <- c(
  "cluster",      # Clustering algorithms
  "factoextra",   # Clustering visualization
  "stringr",      # String manipulation
  "kableExtra"    # Table formatting
)

# Reporting packages
reporting_packages <- c(
  "rmarkdown",    # Report generation
  "knitr",        # Document processing
  "DT"            # Interactive tables
)

cat("Installing packages...\n")
install_packages(c(core_packages, pm_packages, analytics_packages, reporting_packages))
cat("✅ Package installation complete!\n")

## 📥 Step 2: Clone Repository and Load Modules

Clone the HealthProcessAI repository and load the R implementation modules.

In [ ]:
# Clone the repository (when public)
!git clone https://github.com/ki-smile/HealthProcessAI.git
%cd HealthProcessAI

In [ ]:
%%R
# Load all required libraries
suppressPackageStartupMessages({
  library(tidyverse)
  library(lubridate)
  library(glue)
  library(jsonlite)
  library(R6)
  library(httr2)
  library(bupaR)
  library(edeaR)
  library(processmapR)
  library(cluster)
  library(factoextra)
})

cat("✅ Libraries loaded successfully!\n")

## 🔑 Step 3: Configure API Key

Set up your OpenRouter API key for LLM integration. Get your key from [OpenRouter](https://openrouter.ai/).

In [ ]:
%%R
# Set your OpenRouter API key
# Replace 'your-api-key-here' with your actual API key
Sys.setenv(OPENROUTER_API_KEY = "your-api-key-here")

# Verify API key is set
if (nchar(Sys.getenv("OPENROUTER_API_KEY")) > 0) {
  cat("✅ API key configured\n")
} else {
  cat("⚠️ Please set your OpenRouter API key\n")
}

## 📚 Step 4: Load Core R6 Classes

Load the core R6 classes for the 5-step pipeline.

In [ ]:
%%R
# Source core modules
source("R/core/step1_data_loader.R")
source("R/core/step2_process_mining.R")
source("R/core/step3_llm_integration.R")
source("R/core/step4_advanced_analytics.R")
source("R/core/step5_orchestrator.R")
source("R/core/report_generator.R")

cat("✅ Core modules loaded:\n")
cat("  • EventLogLoader (Step 1)\n")
cat("  • ProcessMiner (Step 2)\n")
cat("  • LLMAnalyzer (Step 3)\n")
cat("  • AdvancedProcessAnalyzer (Step 4)\n")
cat("  • ReportOrchestrator (Step 5)\n")
cat("  • ReportGenerator\n")

## 🏥 Step 5: Load and Explore Healthcare Data

Load sepsis event log data and perform initial exploration.

In [ ]:
%%R
# Step 1: Data Loading
cat("\n📊 STEP 1: DATA LOADING\n")
cat("="*50, "\n")

# Initialize data loader
loader <- EventLogLoader$new()

# Load sepsis infection data
event_log <- loader$load_from_csv(
  file_path = "data/sepsisAgregated_Infection.csv",
  case_col = "case",
  activity_col = "activity",
  timestamp_col = "timestamp",
  resource_col = "resource"
)

# Display basic statistics
cat("\n📈 Event Log Statistics:\n")
cat(glue("  • Total events: {nrow(event_log)}\n"))
cat(glue("  • Unique cases: {n_distinct(event_log$case)}\n"))
cat(glue("  • Unique activities: {n_distinct(event_log$activity)}\n"))
cat(glue("  • Date range: {min(event_log$timestamp)} to {max(event_log$timestamp)}\n"))

# Show top activities
cat("\n🏆 Top 5 Activities:\n")
top_activities <- event_log %>%
  count(activity, sort = TRUE) %>%
  head(5)
print(top_activities)

## ⚙️ Step 6: Process Mining Analysis

Discover process patterns using bupaR.

In [ ]:
%%R
# Step 2: Process Mining
cat("\n⚙️ STEP 2: PROCESS MINING\n")
cat("="*50, "\n")

# Initialize process miner
miner <- ProcessMiner$new()

# Discover process model
cat("\n🔍 Discovering process patterns...\n")
dfg <- miner$discover_dfg(event_log, type = "frequency")

# Calculate performance metrics
metrics <- miner$calculate_performance_metrics(event_log)

cat("\n📊 Performance Metrics:\n")
cat(glue("  • Average case duration: {round(metrics$avg_case_duration, 2)} days\n"))
cat(glue("  • Number of variants: {metrics$n_variants}\n"))
cat(glue("  • Most common variant frequency: {round(metrics$most_common_variant_freq*100, 1)}%\n"))

# Identify bottlenecks
bottlenecks <- miner$identify_bottlenecks(event_log, threshold_percentile = 75)
cat("\n🚨 Top Bottlenecks:\n")
print(head(bottlenecks, 5))

## 🤖 Step 7: LLM Analysis

Generate clinical insights using multiple LLM models.

In [ ]:
%%R
# Step 3: LLM Integration
cat("\n🤖 STEP 3: LLM INTEGRATION\n")
cat("="*50, "\n")

# Initialize LLM analyzer
api_key <- Sys.getenv("OPENROUTER_API_KEY")

if (nchar(api_key) > 0) {
  llm_analyzer <- LLMAnalyzer$new(api_key = api_key)
  
  # Prepare process mining results for LLM
  process_summary <- glue("
    Process Mining Analysis Summary:
    - Total cases analyzed: {metrics$n_cases}
    - Average case duration: {round(metrics$avg_case_duration, 2)} days
    - Number of unique process variants: {metrics$n_variants}
    - Most common activities: {paste(top_activities$activity[1:3], collapse = ', ')}
    - Key bottlenecks identified in transitions
  ")
  
  # Query a model for insights
  cat("\n🔮 Generating clinical insights...\n")
  
  clinical_prompt <- "
    Based on this sepsis patient journey analysis, provide:
    1. Key clinical patterns observed
    2. Risk factors for sepsis progression
    3. Recommendations for care optimization
  "
  
  # Query Claude model
  response <- llm_analyzer$query_model(
    model = "anthropic/claude-3.5-sonnet",
    prompt = paste(process_summary, clinical_prompt),
    max_tokens = 500
  )
  
  cat("\n📝 Clinical Insights from Claude:\n")
  cat(response$content)
} else {
  cat("⚠️ No API key configured. Skipping LLM analysis.\n")
  cat("   Set OPENROUTER_API_KEY to enable this feature.\n")
}

## 📊 Step 8: Advanced Analytics

Perform clustering, conformance checking, and predictive analysis.

In [ ]:
%%R
# Step 4: Advanced Analytics
cat("\n📊 STEP 4: ADVANCED ANALYTICS\n")
cat("="*50, "\n")

# Initialize advanced analyzer
advanced_analyzer <- AdvancedProcessAnalyzer$new(event_log)

# Perform patient clustering
cat("\n🔬 Patient Pathway Clustering...\n")
clustering_results <- advanced_analyzer$cluster_patient_pathways(n_clusters = 3)

cat(glue("  • Identified {length(unique(clustering_results$clusters$cluster))} patient clusters\n"))
cat("  • Cluster sizes:\n")
cluster_sizes <- table(clustering_results$clusters$cluster)
for (i in seq_along(cluster_sizes)) {
  cat(glue("    - Cluster {i}: {cluster_sizes[i]} patients\n"))
}

# Calculate clinical KPIs
cat("\n📈 Clinical KPIs:\n")
kpis <- advanced_analyzer$calculate_clinical_kpis()
cat(glue("  • Average length of stay: {round(kpis$avg_los, 2)} days\n"))
cat(glue("  • Case completion rate: {round(kpis$completion_rate*100, 1)}%\n"))
cat(glue("  • Activity efficiency: {round(kpis$activity_efficiency*100, 1)}%\n"))

# Predictive monitoring
cat("\n🎯 Predictive Monitoring:\n")
predictions <- advanced_analyzer$predictive_monitoring(
  target_activity = "Sepsis",
  threshold = 0.3
)
high_risk <- sum(predictions$risk_score > 0.5, na.rm = TRUE)
cat(glue("  • High-risk cases identified: {high_risk}\n"))
cat(glue("  • Average risk score: {round(mean(predictions$risk_score, na.rm = TRUE), 3)}\n"))

## 🎭 Step 9: Multi-Model Orchestration

Orchestrate insights from multiple LLM models.

In [ ]:
%%R
# Step 5: Report Orchestration
cat("\n🎭 STEP 5: MULTI-MODEL ORCHESTRATION\n")
cat("="*50, "\n")

if (exists("llm_analyzer") && nchar(api_key) > 0) {
  # Initialize orchestrator
  orchestrator <- ReportOrchestrator$new()
  
  # Prepare comprehensive process summary
  comprehensive_summary <- list(
    process_metrics = metrics,
    bottlenecks = head(bottlenecks, 5),
    clustering = clustering_results$summary,
    kpis = kpis,
    predictions = list(
      high_risk_count = high_risk,
      avg_risk = mean(predictions$risk_score, na.rm = TRUE)
    )
  )
  
  # Query multiple models
  cat("\n🤖 Querying multiple AI models...\n")
  models <- c(
    "anthropic/claude-3.5-sonnet",
    "openai/gpt-4o",
    "google/gemini-pro-1.5"
  )
  
  multi_model_results <- llm_analyzer$analyze_with_multiple_models(
    process_results = comprehensive_summary,
    models = models[1:2],  # Use first 2 models for demo
    clinical_context = "Sepsis progression analysis"
  )
  
  # Orchestrate results
  cat("\n📋 Orchestrating model insights...\n")
  orchestrated_report <- orchestrator$orchestrate_reports(
    model_results = multi_model_results,
    process_data = comprehensive_summary
  )
  
  cat("\n✅ Orchestration Summary:\n")
  cat(glue("  • Models consulted: {length(multi_model_results)}\n"))
  cat(glue("  • Consensus points: {length(orchestrated_report$consensus)}\n"))
  cat(glue("  • Unique insights: {length(orchestrated_report$unique_insights)}\n"))
  
} else {
  cat("⚠️ Skipping orchestration (requires API key)\n")
}

## 📝 Step 10: Generate Report

Create a comprehensive report with all findings.

In [ ]:
%%R
# Generate comprehensive report
cat("\n📝 GENERATING COMPREHENSIVE REPORT\n")
cat("="*50, "\n")

# Initialize report generator
report_gen <- ReportGenerator$new()

# Prepare report content
report_content <- list(
  title = "Sepsis Patient Journey Analysis",
  date = Sys.Date(),
  summary = "Comprehensive analysis of sepsis progression patterns using process mining and AI",
  sections = list(
    data_overview = list(
      total_cases = n_distinct(event_log$case),
      total_events = nrow(event_log),
      date_range = paste(min(event_log$timestamp), "to", max(event_log$timestamp))
    ),
    process_discovery = list(
      variants = metrics$n_variants,
      avg_duration = round(metrics$avg_case_duration, 2),
      bottlenecks = head(bottlenecks, 3)
    ),
    advanced_analytics = list(
      clusters = length(unique(clustering_results$clusters$cluster)),
      high_risk_cases = high_risk,
      kpis = kpis
    )
  )
)

# Generate markdown report
markdown_report <- report_gen$generate_markdown_report(
  results = report_content,
  include_visualizations = FALSE  # Skip for notebook demo
)

# Display report preview
cat("\n📄 Report Preview:\n")
cat("="*50, "\n")
cat(substr(markdown_report, 1, 500), "...\n")

cat("\n✅ Report generation complete!\n")
cat("   Full report would be saved to: ./reports/sepsis_analysis_report.md\n")

## 🧪 Example: Transform Raw Data to Event Log

Demonstrate transformation of raw clinical data to process mining format.

In [ ]:
%%R
# Load transformation example
source("R/examples/example_transform_raw_to_eventlog.R")

cat("\n🔄 RAW DATA TRANSFORMATION EXAMPLE\n")
cat("="*50, "\n")

# Initialize transformer
transformer <- HealthcareDataTransformer$new()

# Create sample PhysioNet-style data
cat("\n📊 Creating synthetic clinical data...\n")
raw_data <- transformer$create_physionet_style_data(n_patients = 5)
cat(glue("  • Generated {nrow(raw_data)} measurements\n"))
cat(glue("  • Patients: {n_distinct(raw_data$Patient_ID)}\n"))

# Transform to event log
cat("\n🔄 Transforming to event log...\n")
transformed_log <- transformer$transform_to_event_log(raw_data)
cat(glue("  • Created {nrow(transformed_log)} events\n"))
cat(glue("  • Event types: {n_distinct(transformed_log$activity)}\n"))

# Show sample events
cat("\n📋 Sample Events:\n")
print(head(transformed_log %>% select(case, timestamp, activity, resource), 10))

## 🏥 Example: Organ Failure Progression Analysis

Analyze organ failure patterns in sepsis patients.

In [ ]:
%%R
# Load organ failure analysis example
source("R/examples/example_physionet_to_organ.R")

cat("\n🫀 ORGAN FAILURE PROGRESSION ANALYSIS\n")
cat("="*50, "\n")

# Initialize organ transformer
organ_transformer <- PhysioNetToOrganTransformer$new()

# Create sample patients with organ dysfunction
cat("\n📊 Creating patient data with organ dysfunction patterns...\n")
sample_patients <- create_sample_patients()
cat(glue("  • Created {length(sample_patients)} patient trajectories\n"))

# Transform to organ failure events
cat("\n🔄 Detecting organ failure progression...\n")
organ_log <- organ_transformer$create_aggregated_organ_log(sample_patients)

# Analyze patterns
patterns <- organ_transformer$analyze_organ_patterns(organ_log)
cat("\n📈 Organ Failure Patterns:\n")
cat(glue("  • Total cases: {patterns$total_cases}\n"))
cat(glue("  • Sepsis cases: {patterns$sepsis_cases}\n"))
cat(glue("  • Single organ failures: {patterns$organ_failure_types$single_organ}\n"))
cat(glue("  • Multi-organ failures: {patterns$organ_failure_types$multiorgan}\n"))

# Show activity distribution
cat("\n🏥 Organ Failure Events:\n")
activity_dist <- organ_log %>%
  count(activity, sort = TRUE) %>%
  head(5)
print(activity_dist)

- 🏥 [SMAILE Lab](https://smile.ki.se)